# 09 · Structured Outputs (the model as a parser you can trust)

**Where we are in the stack:** the **data plane between model and machine**. So far the
model's *final answer* was prose for a human. The moment another *system* consumes it - a
ticket API, a dashboard, the next agent - prose is a liability.

The networking version of this lesson is old: **screen-scraping CLI output vs a YANG model
over gNMI**. Scraping `show interface` with regexes works until one firmware release moves a
column. The fix was a *schema by contract*. Same move here:

```
free text  ->  "please return JSON" (hope)  ->  JSON mode (valid JSON, wrong shape?)
           ->  schema + validation + retry  (a contract)
```

We take messy `show interface` text and turn it into a **validated Pydantic object** -
guaranteed fields, guaranteed types - with a retry loop for when the model gets it wrong.

> Works on any chat model; no tool-calling needed. Needs `pydantic` (in `requirements.txt`).

In [ ]:
# --- Provider config: works with OpenAI, OpenRouter, or a local OpenAI-compatible server ---
import os
from openai import OpenAI

# Load settings from a .env file if present (falls back to existing env vars).
try:
    from dotenv import load_dotenv, find_dotenv
    load_dotenv(find_dotenv(usecwd=True))
except Exception:
    import os
    if os.path.exists(".env"):
        for _line in open(".env"):
            _line = _line.strip()
            if _line and not _line.startswith("#") and "=" in _line:
                _k, _v = _line.split("=", 1)
                os.environ.setdefault(_k.strip(), _v.strip())


# Pick ONE setup by exporting these env vars before launching Jupyter.
#
#   OpenAI:     OPENAI_BASE_URL=https://api.openai.com/v1   MODEL=gpt-4o-mini
#   OpenRouter: OPENAI_BASE_URL=https://openrouter.ai/api/v1 MODEL=openai/gpt-4o-mini
#   Local:      OPENAI_BASE_URL=http://localhost:11434/v1    MODEL=qwen2.5:7b   (Ollama)
#               (use 'qwen2.5' / 'llama3.1' etc. - a 1B model is great for chat but
#                usually too weak to drive tool-calling reliably.)

BASE_URL = os.environ.get("OPENAI_BASE_URL", "https://api.openai.com/v1")
API_KEY  = os.environ.get("OPENAI_API_KEY", "set-me")   # any non-empty string for local servers
MODEL    = os.environ.get("MODEL", "gpt-4o-mini")

# Behind a TLS-intercepting firewall/proxy, HTTPS cert verification can fail.
# Set VERIFY_SSL=false in .env to skip it: we hand the OpenAI SDK a custom
# httpx client with verification turned off. Leave it true everywhere else.
import httpx
VERIFY_SSL = os.environ.get("VERIFY_SSL", "true").strip().lower() not in ("false", "0", "no")
http_client = httpx.Client(verify=VERIFY_SSL)
if not VERIFY_SSL:
    import warnings
    warnings.filterwarnings("ignore")
    print("\u26a0\ufe0f  SSL verification DISABLED (VERIFY_SSL=false) \u2014 use only on a trusted network")

client = OpenAI(base_url=BASE_URL, api_key=API_KEY, http_client=http_client)
print("endpoint:", BASE_URL, "| model:", MODEL)

## 1. The messy input

Real CLI output: inconsistent spacing, vendor phrasing, the interesting facts buried mid-line.
Exactly what you would otherwise attack with a pile of regexes.

In [ ]:
RAW = """
leaf-01# show interface ethernet1/0/1
Ethernet1/0/1 is up, line protocol is down (err-disabled)
  Hardware is 10G-SFP+, address is 00a1.b2c3.d4e5
  MTU 9000 bytes, BW 10000000 Kbit
  Last clearing of counters: 3d 02:11:45
  30 second input rate 0 bits/sec, 0 packets/sec
  RX: 184023 unicast packets, 402 multicast
      1732 input errors, 1698 CRC, 34 frame
  TX: 220119 unicast packets, 87 multicast
      0 output errors, 0 collisions
"""
print(RAW)

## 2. The naive way: ask for JSON and hope

No schema, no guarantees. Typical failure modes you will meet in the wild:
- the JSON arrives wrapped in a ` ```json ` fence, or with a "Here is the JSON:" preamble
- a numeric field comes back as a string (`"1698"`), or a key is spelled differently each run
- fields you never asked for appear; fields you need are missing

It often *looks* fine - which is worse, because the parser downstream breaks at 3am, not in
the demo.

In [ ]:
resp = client.chat.completions.create(
    model=MODEL, temperature=0,
    messages=[{"role": "user",
               "content": "Extract the key facts from this CLI output as JSON:\n" + RAW}],
)
print(resp.choices[0].message.content)

## 3. Define the contract (a Pydantic model)

This class is the YANG model of our pipeline: field names, types, and allowed values,
declared once. `Literal` pins enums (`oper_status` cannot be `"kinda up"`), validators can
enforce cross-field rules, and `model_json_schema()` renders the whole thing as JSON Schema
we can hand to the model.

In [ ]:
from typing import Literal
from pydantic import BaseModel, Field

class InterfaceReport(BaseModel):
    """Validated summary of one 'show interface' output."""
    device: str = Field(description="hostname the CLI was captured on, e.g. leaf-01")
    interface: str = Field(description="interface name, e.g. Ethernet1/0/1")
    admin_status: Literal["up", "down"]
    oper_status: Literal["up", "down", "err-disabled"]
    mtu: int
    input_errors: int
    crc_errors: int
    output_errors: int
    is_healthy: bool = Field(description="true only if oper_status is 'up' and error counters are zero")
    issues: list[str] = Field(description="short human-readable list of anything wrong; empty if healthy")

import json
print(json.dumps(InterfaceReport.model_json_schema(), indent=2)[:500], "...")

## 4. JSON mode + validation + retry

Three layers, each catching what the previous one cannot:

1. **JSON mode** (`response_format={"type": "json_object"}`) - the provider guarantees the
   reply parses as JSON. Shape and types are still *not* guaranteed.
2. **Pydantic validation** - the contract check. Wrong type, missing field, illegal enum
   value -> a precise `ValidationError`.
3. **Retry with the error** - feed the validation error back; the model corrects itself.
   This is a *negotiation*, like a BGP session retrying with the right capabilities.

(Many providers now ship this bundle natively - e.g. `client.beta.chat.completions.parse`
or Anthropic/OpenAI "structured outputs" modes. Under the hood it is this same contract;
we hand-roll it once so you know what those APIs do for you.)

In [ ]:
from pydantic import ValidationError

def extract(raw_text, max_attempts=3, temperature=0):
    """Messy CLI text -> validated InterfaceReport (or raise after max_attempts)."""
    schema = json.dumps(InterfaceReport.model_json_schema(), indent=2)
    messages = [
        {"role": "system", "content":
            "You extract structured data from network CLI output. "
            "Reply with ONE JSON object that satisfies this JSON Schema exactly - "
            "no extra keys, no prose:\n" + schema},
        {"role": "user", "content": raw_text},
    ]

    for attempt in range(1, max_attempts + 1):
        resp = client.chat.completions.create(
            model=MODEL, messages=messages, temperature=temperature,
            response_format={"type": "json_object"},          # layer 1: valid JSON
        )
        content = resp.choices[0].message.content
        try:
            report = InterfaceReport.model_validate_json(content)   # layer 2: valid shape
            print(f"[attempt {attempt}] validated OK")
            return report
        except ValidationError as e:
            print(f"[attempt {attempt}] validation failed:\n{e}\n-> retrying with the error")
            messages.append({"role": "assistant", "content": content})
            messages.append({"role": "user", "content":                 # layer 3: negotiate
                "That JSON failed validation with the error below. "
                "Return a corrected JSON object.\n" + str(e)})

    raise RuntimeError(f"could not produce a valid InterfaceReport in {max_attempts} attempts")

## 5. Run it

The result is not a string - it is an object. `report.crc_errors` is an `int` you can
compare, sum, and alert on. This is the boundary where LLM output becomes safe to feed
into ordinary software.

In [ ]:
report = extract(RAW)
print()
print(json.dumps(report.model_dump(), indent=2))
print()
print("healthy?", report.is_healthy, "| crc:", report.crc_errors, "| issues:", report.issues)

## 6. Break it on purpose

Two stress tests:
- **A second, different-looking capture** (different vendor phrasing) - the schema, not a
  regex, absorbs the difference.
- **Garbage input** - the contract still forces *some* legal shape, which is exactly when you
  add an application-level check: does the output actually correspond to the input? (A
  validator can only prove the shape is legal, not that the facts are right - keep that
  distinction in mind for notebook 12.)

In [ ]:
RAW2 = """
spine-02> sh int et-0/0/7 | no-more
Physical interface: et-0/0/7, Enabled, Physical link is Up
  MTU: 1514, Speed: 40Gbps
  Input errors: 0, CRC errors: 0
  Output errors: 0
  Interface flaps: 0
"""
r2 = extract(RAW2)
print(json.dumps(r2.model_dump(), indent=2))

## Recap

- Prose is for humans; **schemas are for systems**. The moment an LLM output feeds an API,
  a database, or another agent, define the contract first.
- The reliable stack is three thin layers: **JSON mode -> Pydantic validation -> retry with
  the error**. Each catches a failure class the previous one cannot.
- Validation proves the *shape* is legal, not that the *facts* are true - evaluation
  (notebook 12) covers the second half.
- Provider-native structured output APIs are this exact bundle, pre-packaged. Use them in
  production; now you know what they are doing.

This also closes a loop with notebook 02: tool *calls* were already structured (the model
emitting JSON arguments against a schema). Today we applied the same discipline to tool
*answers* and final outputs.

**Next:** giving the agent a memory that survives the turn - and managing the context window
it lives in. -> `10_memory_and_context.ipynb`